# 📊 Data Profiling & Quality Audit: Brazilian E-Commerce (Olist)

**Penyusun:** Anggota 2 (Data Quality & Cleansing Engineer)  
**Proyek:** Data Warehouse & AI-assisted ETL  
**Tujuan:** Mengaudit kualitas data mentah (*raw data*), memeriksa sebaran *missing values*, duplikasi data, konsistensi tipe data, serta anomali sebelum dilakukan transformasi pembersihan data (*cleansing*).

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

RAW_DIR = '../../data/raw'
print(f'Membaca direktori data mentah: {os.path.abspath(RAW_DIR)}')

## 1. Audit Ringkasan Dimensi & Missing Values

In [ ]:
csv_files = sorted(glob.glob(os.path.join(RAW_DIR, '*.csv')))
summary_records = []

for f in csv_files:
    filename = os.path.basename(f)
    df = pd.read_csv(f)
    summary_records.append({
        'File': filename,
        'Total Baris': len(df),
        'Total Kolom': len(df.columns),
        'Duplikasi': df.duplicated().sum(),
        'Kolom Ada Null': (df.isnull().sum() > 0).sum()
    })

pd.DataFrame(summary_records)

## 2. Pengecekan Kategori Produk & Terjemahan

In [ ]:
products_df = pd.read_csv(os.path.join(RAW_DIR, 'olist_products_dataset.csv'))
trans_df = pd.read_csv(os.path.join(RAW_DIR, 'product_category_name_translation.csv'))

missing_cats = products_df['product_category_name'].isnull().sum()
diff_cats = set(products_df['product_category_name'].dropna()) - set(trans_df['product_category_name'])

print(f'Jumlah produk tanpa nama kategori: {missing_cats}')
print(f'Kategori di tabel produk tapi belum ada di translation: {diff_cats}')

## 3. Pengecekan Integritas Tanggal Pesanan (Orders)

In [ ]:
orders_df = pd.read_csv(os.path.join(RAW_DIR, 'olist_orders_dataset.csv'))
print('Distribusi Status Pesanan:')
print(orders_df['order_status'].value_counts())

print('\nMissing values pada kolom tanggal:')
date_cols = [c for c in orders_df.columns if 'date' in c or 'timestamp' in c or 'at' in c]
print(orders_df[date_cols].isnull().sum())

## 4. Pengecekan Pembayaran (Payments) & Ulasan (Reviews)

In [ ]:
payments_df = pd.read_csv(os.path.join(RAW_DIR, 'olist_order_payments_dataset.csv'))
print('Distribusi Tipe Pembayaran:')
print(payments_df['payment_type'].value_counts())

reviews_df = pd.read_csv(os.path.join(RAW_DIR, 'olist_order_reviews_dataset.csv'))
print('\nDistribusi Skor Ulasan (Review Score):')
print(reviews_df['review_score'].value_counts().sort_index())